In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from combra import data
from combra.metrics import wdist_vs_n
from combra.metrics.compare import load_rows

types_dict = {
    'Ultra_Co11': 'medium grain',
    'Ultra_Co25': 'small grain',
    'Ultra_Co8':  'medium-small grain',
    'Ultra_Co6_2': 'large grain',
    'Ultra_Co15': 'medium-small grain',
}

# diffit and san were trained on 3 classes; map their class_0/1/2 outputs to the ethalon class names.
class_map = {
    'class_0': 'class_Ultra_Co11',
    'class_1': 'class_Ultra_Co25',
    'class_2': 'class_Ultra_Co6_2',
}

# Per-resolution sources for the 3x3 convergence grid.
# Each row = one resolution; the 'real' H5 doubles as ethalon (largest-N parquet)
# and as the source of the original-data self-convergence curve.
# Originals are the 1024-source crops in data/generated_images_h5/.
# 1024 row has only the original — no san/gen_diff/diffit at 1024.
SOURCES = {
    256: {
        'real':     './data/generated_images_h5/o_bc_left_4x_1536_1024x1024_256x256_rgb_N360.h5',
        'san':      './data/generated_images_h5/gen_san_256x256_N100_000.h5',
        'gen_diff': './data/generated_images_h5/gen_diff_256x256_N500.h5',
        'diffit':   './data/generated_images_h5/00017-diffit-256-gpus2-batch192_kimg_006451.h5',
    },
    512: {
        'real':     './data/generated_images_h5/o_bc_left_4x_1536_1024x1024_512x512_rgb_N360.h5',
        'san':      './data/generated_images_h5/gen_san_512x512_N100_000.h5',
        'gen_diff': './data/generated_images_h5/gen_diff_512x512_N500.h5',
    },
    1024: {
        'real':     './data/generated_images_h5/o_bc_left_4x_1536_1024x1024_1024x1024_rgb_N360.h5',
    },
}

# N sweep values per source kind. Bounded by the dataset size of each H5.
N_VALUES = {
    'real':     [50, 100, 150, 200, 250, 300, 360],
    'san':      [100, 250, 500, 1000, 2500, 5000, 10000],
    'gen_diff': [50, 100, 250, 500],
    'diffit':   [100, 250, 500, 1000],
}

# Columns of the 3x3 grid — the 3 grain classes diffit was trained on.
CLASSES = ['class_Ultra_Co11', 'class_Ultra_Co25', 'class_Ultra_Co6_2']

LABELS = {
    'real':     'original',
    'san':      'san',
    'gen_diff': 'vanilla diff',
    'diffit':   'diffit ',
}

# Kinds whose parquets use class_0/1/2 and need class_map applied.
NUMERIC_CLASS_KINDS = ('san', 'diffit')

ANGLES_ROOT = Path('./data/angles')

def sweep_dir(h5_path):
    return ANGLES_ROOT / (Path(h5_path).stem + '_msl5')

def parquet_for(h5_path, n):
    return sweep_dir(h5_path) / f'angles_n{n}.parquet'

# Generate cap-N angle parquets for every source

One pass over each H5 in `SOURCES` capped at `max(N_VALUES[kind])`. Output:
`./data/angles/<h5-stem>_msl5/angles_n{cap}.parquet`. Skipped if the parquet
already exists. The cap-N parquet of each *real* folder doubles as the ethalon
for that resolution row in the 3×3 plot.

Some H5s (san at 100k images) are too large to process in full, hence the cap.
If `N_VALUES[kind]` is changed later, re-run this cell to refresh the cap-N
parquet at the new top.

In [ ]:
for resolution, group in SOURCES.items():
    print(f'\n=== resolution {resolution}x{resolution} ===')
    for kind, h5_path in group.items():
        out_dir = sweep_dir(h5_path)
        cap = max(N_VALUES[kind])
        out_pq = out_dir / f'angles_n{cap}.parquet'
        if out_pq.exists():
            print(f'  [{kind}] cap-N parquet present ({out_pq.name})')
            continue
        print(f'  [{kind}] {h5_path} -> {out_pq}, generating cap N={cap}')
        ds = data.PobeditDataset(path=h5_path, max_images_num_per_class=cap)
        ds.generate_angles(
            save_path=str(out_dir),
            types_dict=types_dict,
            step=[5.0],
            workers=20,
            angles_tol=3,
            min_segment_len=5.0,
            keep_contours=False,
            chunksize=64,
        )

# N-sweep generation for the convergence study

For every (resolution, source) entry in `SOURCES` extract angle distributions
at every N value listed in `N_VALUES[kind]`. Output convention is the same as
the full-N cell: `./data/angles/<h5-stem>_msl5/angles_n{N}.parquet`. Idempotent
— already-existing parquets are skipped, so trim `N_VALUES` while iterating.

For san at 100k images this is the slowest step. Originals reuse existing
`prep_cache_*_n360.npy` files in the H5 folder, so N=360 is fast.

In [ ]:
for resolution, group in SOURCES.items():
    print(f'\n=== resolution {resolution}x{resolution} ===')
    for kind, h5_path in group.items():
        out_dir = sweep_dir(h5_path)
        ns = N_VALUES[kind]
        missing = [n for n in ns if not parquet_for(h5_path, n).exists()]
        if not missing:
            print(f'  [{kind}] all N present in {out_dir}')
            continue
        print(f'  [{kind}] {h5_path} -> {out_dir}, generating N={missing}')
        for N in missing:
            ds = data.PobeditDataset(path=h5_path, max_images_num_per_class=N)
            ds.generate_angles(
                save_path=str(out_dir),
                types_dict=types_dict,
                step=[5.0],
                workers=20,
                angles_tol=3,
                min_segment_len=5.0,
                keep_contours=False,
                chunksize=64,
            )

# 3×3 W-dist convergence grid

Rows = resolution (256, 512, 1024). Columns = grain class. Each cell overlays
the W-dist-vs-N curve for the original (self-convergence baseline against the
full-N ethalon at that resolution) plus each generator available at that
resolution.

The 1024 row currently has only the original baseline — no san at 1024 in the
data folder, no gen_diff at 1024 either. Diffit appears in the 256 row only.

In [ ]:
def _max_n(parquet_path):
    rows = load_rows(parquet_path)
    return max((int(r['meta']['n_images']) for r in rows), default=0)

WDIST_SCALE = 1000

# Human-friendly grain labels for subplot titles.
GRAIN_LABEL = {
    'class_Ultra_Co11':  'medium grain',
    'class_Ultra_Co25':  'small grain',
    'class_Ultra_Co6_2': 'large grain',
}

fig, axes = plt.subplots(len(SOURCES), len(CLASSES),
                         figsize=(6 * len(CLASSES), 5 * len(SOURCES)),
                         sharex=True, sharey=True, squeeze=False)

fig.suptitle(f'W-dist convergence (step=5, ×{WDIST_SCALE})', fontsize=17, y=0.995)

for row, (resolution, group) in enumerate(SOURCES.items()):
    real_dir = sweep_dir(group['real'])
    real_parquets = sorted(real_dir.glob('*.parquet'))
    if not real_parquets:
        print(f'[row {resolution}] no ethalon parquets in {real_dir} — skip row')
        continue
    ethalon_path = max(real_parquets, key=_max_n)
    print(f'row {resolution}: ethalon = {ethalon_path}')

    src_records = {}
    for kind, h5 in group.items():
        d = sweep_dir(h5)
        if not d.exists():
            continue
        cmap = class_map if kind in NUMERIC_CLASS_KINDS else None
        recs = wdist_vs_n(d, str(ethalon_path), class_map=cmap, step=5.0)
        if kind == 'real':
            recs = [r for r in recs if r['w_dist'] > 0]
        src_records[kind] = recs

    for col, cls in enumerate(CLASSES):
        ax = axes[row, col]
        for kind, recs in src_records.items():
            pts = sorted((r['n_images'], r['w_dist']) for r in recs if r['class'] == cls)
            if not pts:
                continue
            Ns, ws = zip(*pts)
            ax.plot(Ns, [w * WDIST_SCALE for w in ws], '-o',
                    label=LABELS.get(kind, kind), markersize=8, linewidth=2)

        ax.set_xscale('log')
        ax.grid(True, which='both', linestyle='-', linewidth=0.6, alpha=0.35)
        ax.set_axisbelow(True)
        ax.tick_params(axis='both', labelsize=13)
        ax.set_title(f'{resolution}×{resolution} — {GRAIN_LABEL.get(cls, cls)}', fontsize=15)
        if row == len(SOURCES) - 1:
            ax.set_xlabel('Number of images, N', fontsize=14)
        if col == 0:
            ax.set_ylabel(f'W-dist × {WDIST_SCALE}', fontsize=14)
        ax.legend(fontsize=12, loc='lower right', framealpha=0.9)

plt.tight_layout(rect=(0, 0, 1, 0.985))
plt.show()